# Cross-Campaign Optimization Filter Check

Collect the latest simulation pickles across all campaigns and operating modes, then evaluate whether each run passes the optimization filter criteria.

**Outputs**
1. Detailed pass/fail table (one row per selected simulation)
2. Optional summary pass-rate table by campaign and mode
3. CSV export(s)

## 1. Setup & Configuration

In [1]:
from __future__ import annotations

import pickle
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd

from animakinetica.core.simulation import CONTINUOUS_MODES

# Paths
NOTEBOOK_DIR = Path().resolve()
ROOT_DIR = NOTEBOOK_DIR.parent

# Discovery and selection config
CAMPAIGN_GLOB = "Campaign_*"
EXCLUDE_PATH_PARTS = {"06_Analysis", "Cross_Campaign_Comparison"}
SKIP_SENSITIVITY_PICKLES = True
KEEP_LATEST_ONLY = True

# Select "s1", "s2", "undefined" (legacy _opt_ exports), or "all"/None.
# Latest-file selection is performed separately for each campaign/mode/run/sweep.
SWEEP_FILTER = "all"

# Export config
EXPORT_DETAILED_CSV = False
EXPORT_SUMMARY_CSV = False
DETAILED_CSV_NAME = "cross_campaign_optimization_filter_results.csv"
SUMMARY_CSV_NAME = "cross_campaign_optimization_filter_summary.csv"

# Filter thresholds (mirror optimization.py).
UNWANTED_TERMINATION_REASONS = {"cell minimum", "unknown termination"}
CONTINUOUS_ALLOWED_TERMINATION_REASONS = {"maximum time reached"}
CONTINUOUS_MIN_RUNTIME_FRACTION = 0.995
SIMULATION_WINDOW_MIN_RUNTIME_FRACTION = 0.995
MIN_FINAL_VOLUME_FRACTION = 0.98
MAX_ACID_BASE_FRACTION = 0.05
MAX_CLOSURE_PCT = 5.0

print(f"Root directory: {ROOT_DIR}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Configured sweep filter: {SWEEP_FILTER!r}")

Root directory: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code
Notebook directory: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Cross_Campaign_Comparison
Configured sweep filter: 'all'


In [2]:
# Helpers: discovery, folder-metadata parsing, and latest selection

EXPORT_FOLDER_RE = re.compile(
    r"^(?P<run_key>.+?)_(?P<sweep_token>s1|s2|p1|p2|opt)_(?P<date>\d{8})_(?P<time>\d{4})$",
    re.IGNORECASE,
)
VALID_SWEEP_FILTERS = {"s1", "s2", "p1","p2", "undefined", "all", None}


def canonical_mode(mode_raw: str) -> str:
    s = str(mode_raw).lower()
    s = re.sub(r"^\d+[_\-\s]*", "", s)
    s = s.replace("_", "").replace("-", "").replace(" ", "")
    if "concentratedfedbatch" in s or s.startswith("cf"):
        return "Concentrated FedBatch"
    if "continuousperfusion" in s or s.startswith("cp"):
        return "Continuous Perfusion"
    if "fedbatch" in s:
        return "FedBatch"
    if "chemostat" in s:
        return "Chemostat"
    if "batch" in s:
        return "Batch"
    return str(mode_raw)


def campaign_folder_from_path(pkl_path: Path, root_dir: Path) -> Optional[Path]:
    """Return the campaign directory containing a pickle, based on its folder name."""
    for parent in (pkl_path.parent, *pkl_path.parents):
        if parent.parent == root_dir and parent.name.lower().startswith("campaign_"):
            return parent
    return None


def mode_folder_from_path(pkl_path: Path, campaign_dir: Path) -> str:
    """Return the numbered mode folder directly beneath the campaign directory."""
    try:
        relative_parts = pkl_path.relative_to(campaign_dir).parts
    except ValueError:
        relative_parts = pkl_path.parts
    return next(
        (part for part in relative_parts if re.match(r"^\d{2}_", part)),
        pkl_path.parent.name,
    )


def parse_pickle_metadata(pkl_path: Path, root_dir: Path) -> dict[str, Any]:
    """Parse campaign, mode, run, sweep, and timestamp from export-folder naming."""
    campaign_dir = campaign_folder_from_path(pkl_path, root_dir)
    if campaign_dir is None:
        raise ValueError(f"Could not identify campaign folder for {pkl_path}")

    campaign_match = re.search(r"campaign[_\-]?(\d+)", campaign_dir.name, re.IGNORECASE)
    campaign = f"Campaign_{campaign_match.group(1)}" if campaign_match else campaign_dir.name
    mode_folder = mode_folder_from_path(pkl_path, campaign_dir)
    export_folder = pkl_path.parent.name
    match = EXPORT_FOLDER_RE.match(export_folder)

    if match:
        raw_sweep_token = match.group("sweep_token").lower()
        sweep = "undefined" if raw_sweep_token == "opt" else raw_sweep_token
        run_id = match.group("run_key")
        export_timestamp_text = f"{match.group('date')}_{match.group('time')}"
        export_timestamp = datetime.strptime(export_timestamp_text, "%Y%m%d_%H%M")
    else:
        raw_sweep_token = "unknown"
        sweep = "unknown"
        run_id = pkl_path.stem
        export_timestamp_text = ""
        export_timestamp = None

    return {
        "campaign_dir": campaign_dir,
        "campaign": campaign,
        "mode_folder": mode_folder,
        "mode_group": canonical_mode(mode_folder),
        "run_id": run_id,
        "sweep": sweep,
        "raw_sweep_token": raw_sweep_token,
        "export_folder": export_folder,
        "export_timestamp": export_timestamp,
        "export_timestamp_text": export_timestamp_text,
        "relative_path": str(pkl_path.relative_to(root_dir)),
    }


def sweep_matches(metadata: dict[str, Any], sweep_filter: Optional[str]) -> bool:
    if sweep_filter is None or str(sweep_filter).lower() == "all":
        return True
    return metadata["sweep"] == str(sweep_filter).lower()


def is_excluded_path(path: Path) -> bool:
    parts = set(path.parts)
    if parts & EXCLUDE_PATH_PARTS:
        return True
    return SKIP_SENSITIVITY_PICKLES and "_sens_" in path.stem.lower()


def discover_pickles(root_dir: Path) -> list[Path]:
    all_files: list[Path] = []
    for campaign_dir in sorted(root_dir.glob(CAMPAIGN_GLOB)):
        if not campaign_dir.is_dir():
            continue
        for p in campaign_dir.rglob("*.pkl"):
            if not is_excluded_path(p):
                all_files.append(p)
    return sorted(all_files)


def select_latest_pickles(pickle_files: list[Path], root_dir: Path) -> list[Path]:
    """Keep the newest export per campaign, mode, run key, and sweep."""
    if not KEEP_LATEST_ONLY:
        return sorted(pickle_files)

    latest: dict[tuple[str, str, str, str], Path] = {}
    for p in pickle_files:
        metadata = parse_pickle_metadata(p, root_dir)
        key = (
            metadata["campaign"],
            metadata["mode_folder"],
            metadata["run_id"],
            metadata["sweep"],
        )
        if key not in latest or p.stat().st_mtime > latest[key].stat().st_mtime:
            latest[key] = p
    return sorted(latest.values())

## 2. Discover and Select Latest Pickles

In [3]:
all_pkl_files = discover_pickles(ROOT_DIR)
metadata_by_path = {p: parse_pickle_metadata(p, ROOT_DIR) for p in all_pkl_files}
sweep_candidate_files = [p for p in all_pkl_files if sweep_matches(metadata_by_path[p], SWEEP_FILTER)]
selected_files = select_latest_pickles(sweep_candidate_files, ROOT_DIR)

print(f"Pickles discovered: {len(all_pkl_files)}")
print(f"Pickles matching sweep {SWEEP_FILTER!r}: {len(sweep_candidate_files)}")
print(f"Pickles selected (latest per campaign/mode/run/sweep): {len(selected_files)}")

sweep_counts = (
    pd.Series([metadata_by_path[p]["sweep"] for p in all_pkl_files], name="count")
    .value_counts()
    .rename_axis("sweep")
    .reset_index()
)
display(sweep_counts)

selected_set = set(selected_files)
preview_rows = [
    {
        "selected": p in selected_set,
        "campaign": metadata_by_path[p]["campaign"],
        "mode_folder": metadata_by_path[p]["mode_folder"],
        "run_id": metadata_by_path[p]["run_id"],
        "sweep": metadata_by_path[p]["sweep"],
        "export_timestamp": metadata_by_path[p]["export_timestamp"],
        "export_folder": metadata_by_path[p]["export_folder"],
        "relative_path": metadata_by_path[p]["relative_path"],
    }
    for p in sweep_candidate_files
]

preview_df = pd.DataFrame(preview_rows)
if not preview_df.empty:
    preview_df = preview_df.sort_values(
        ["selected", "campaign", "mode_folder", "run_id", "export_timestamp"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)
display(preview_df)

Pickles discovered: 197
Pickles matching sweep 'all': 197
Pickles selected (latest per campaign/mode/run/sweep): 112


,sweep,count
0,p2,187
1,undefined,5
2,s2,5


,selected,campaign,mode_folder,run_id,sweep,export_timestamp,export_folder,relative_path
0,True,Campaign_1,01_FedBatch,fedbatch_constant,p2,2026-08-24 23:51:00,fedbatch_constant_p2_20260824_2351,Campaign_1_fixed_stoichiometry_Humbird\01_FedB...
1,True,Campaign_1,01_FedBatch,fedbatch_csfr,p2,2026-08-25 00:26:00,fedbatch_csfr_p2_20260825_0026,Campaign_1_fixed_stoichiometry_Humbird\01_FedB...
2,True,Campaign_1,01_FedBatch,fedbatch_exponential,p2,2026-08-25 00:59:00,fedbatch_exponential_p2_20260825_0059,Campaign_1_fixed_stoichiometry_Humbird\01_FedB...
3,True,Campaign_1,01_FedBatch,fedbatch_gsp,p2,2026-08-25 05:51:00,fedbatch_gsp_p2_20260825_0551,Campaign_1_fixed_stoichiometry_Humbird\01_FedB...
4,True,Campaign_1,01_FedBatch,fedbatch_linear,p2,2026-08-25 06:33:00,fedbatch_linear_p2_20260825_0633,Campaign_1_fixed_stoichiometry_Humbird\01_FedB...
...,...,...,...,...,...,...,...,...
192,False,Campaign_6,05_ContinuousPerfusion,cp_constant,p2,2026-08-21 01:19:00,cp_constant_p2_20260821_0119,Campaign_6_fixed_stoichiometry_Humbird_enhance...
193,False,Campaign_6,05_ContinuousPerfusion,cp_cspr,p2,2026-08-21 02:42:00,cp_cspr_p2_20260821_0242,Campaign_6_fixed_stoichiometry_Humbird_enhance...
194,False,Campaign_6,05_ContinuousPerfusion,cp_gsp,p2,2026-08-21 08:44:00,cp_gsp_p2_20260821_0844,Campaign_6_fixed_stoichiometry_Humbird_enhance...
195,False,Campaign_6,05_ContinuousPerfusion,cp_tbp,p2,2026-08-21 09:44:00,cp_tbp_p2_20260821_0944,Campaign_6_fixed_stoichiometry_Humbird_enhance...


## 3. Load Selected Simulations

In [4]:
loaded: dict[str, Any] = {}
load_errors: dict[str, str] = {}

for p in selected_files:
    key = str(p.relative_to(ROOT_DIR))
    try:
        with open(p, "rb") as fh:
            loaded[key] = pickle.load(fh)
    except Exception as exc:
        load_errors[key] = f"{type(exc).__name__}: {exc}"

print(f"Loaded objects: {len(loaded)}")
print(f"Load errors: {len(load_errors)}")

if load_errors:
    display(pd.DataFrame(
        [{"relative_path": k, "error": v} for k, v in load_errors.items()]
    ))

Loaded objects: 112
Load errors: 0


In [5]:
# Normalize loaded objects to SimulationData and retain metadata parsed from export folders.

@dataclass
class LoadedItem:
    relative_path: str
    pickle_path: Path
    campaign: str
    mode_folder: str
    run_id: str
    sweep: str
    export_folder: str
    export_timestamp: Optional[datetime]
    mode_group: str
    sim_obj: Any
    data_obj: Any


normalized_items: list[LoadedItem] = []
normalize_errors: list[dict[str, str]] = []

for rel_path, obj in loaded.items():
    p = ROOT_DIR / rel_path
    try:
        data_obj = obj.data if hasattr(obj, "data") else obj
        if not hasattr(data_obj, "simulated_window_totals_stats"):
            raise ValueError("Object does not expose simulated_window_totals_stats")

        metadata = metadata_by_path[p]
        mode_raw = getattr(data_obj, "simulation_type", metadata["mode_folder"])
        normalized_items.append(
            LoadedItem(
                relative_path=rel_path,
                pickle_path=p,
                campaign=metadata["campaign"],
                mode_folder=metadata["mode_folder"],
                run_id=metadata["run_id"],
                sweep=metadata["sweep"],
                export_folder=metadata["export_folder"],
                export_timestamp=metadata["export_timestamp"],
                mode_group=canonical_mode(mode_raw),
                sim_obj=obj if hasattr(obj, "operating_mode") else None,
                data_obj=data_obj,
            )
        )
    except Exception as exc:
        normalize_errors.append({"relative_path": rel_path, "error": f"{type(exc).__name__}: {exc}"})

print(f"Normalized items: {len(normalized_items)}")
print(f"Normalization errors: {len(normalize_errors)}")

if normalize_errors:
    display(pd.DataFrame(normalize_errors))

Normalized items: 112
Normalization errors: 0


## 4. Evaluate Optimization Filter Checks

In [6]:
def evaluate_filter_checks(item: LoadedItem) -> dict[str, Any]:
    """Evaluate the filter criteria that can be reconstructed from saved data.

    This mirrors ``ObjectiveFunction`` in ``optimization.py``. Runtime checks
    intentionally use ``total_simulation_time_h``: active-cultivation-time
    tracking is no longer emitted by simulations.
    """
    data = item.data_obj
    simulated_totals = getattr(data, "simulated_window_totals_stats", {}) or {}

    # Mode handling for continuous checks
    sim_obj = item.sim_obj
    operating_mode = getattr(sim_obj, "operating_mode", None)
    is_continuous = bool(operating_mode in CONTINUOUS_MODES) if operating_mode is not None else False
    max_culture_age = float(getattr(sim_obj, "max_culture_age", np.nan)) if sim_obj is not None else np.nan

    checks: dict[str, bool] = {}

    # 1) Time
    total_time_h = float(simulated_totals.get("total_simulation_time_h", 0.0) or 0.0)
    checks["time_gte_72h"] = total_time_h >= 72.0

    # 2) Unwanted termination
    last_reason = str(simulated_totals.get("last_batch_termination_reason", "")).strip().lower()
    unique_reasons = {
        str(r).strip().lower()
        for r in simulated_totals.get("unique_batch_termination_reasons", [])
        if str(r).strip()
    }
    has_unwanted = bool(unique_reasons & UNWANTED_TERMINATION_REASONS) or (last_reason in UNWANTED_TERMINATION_REASONS)
    checks["no_unwanted_termination"] = not has_unwanted

    # 3) All-batch final-volume criterion. This is a soft constraint in the
    # optimizer, but it is shown as a failed filter check in this report.
    min_final_volume_fraction = simulated_totals.get("batch_final_volume_min_fraction", None)
    if min_final_volume_fraction is None:
        min_final_volume_fraction = simulated_totals.get("last_batch_final_volume_fraction", 0.0)
    try:
        min_final_volume_fraction = float(min_final_volume_fraction)
    except (TypeError, ValueError):
        min_final_volume_fraction = np.nan

    checks["all_batches_final_volume_ok"] = (
        np.isfinite(min_final_volume_fraction)
        and min_final_volume_fraction >= MIN_FINAL_VOLUME_FRACTION
    )

    # 4) All-mode simulated-window runtime completeness. ``optimization.py``
    # compares total_simulation_time_h with the raw temporary-run horizon.
    def _expected_simulation_window_target_h(simulation_obj: Any) -> float:
        max_culture_age_h = float(getattr(simulation_obj, "max_culture_age", 0.0) or 0.0)
        yearly_uptime_h = float(getattr(simulation_obj, "yearly_uptime", 0.0) or 0.0)
        if max_culture_age_h > 0.0 and yearly_uptime_h > 0.0:
            return min(max_culture_age_h, yearly_uptime_h)
        if max_culture_age_h > 0.0:
            return max_culture_age_h
        if yearly_uptime_h > 0.0:
            return yearly_uptime_h
        return np.nan

    expected_window_target_h = _expected_simulation_window_target_h(sim_obj)
    checks["all_mode_full_window_runtime"] = (
        np.isfinite(expected_window_target_h)
        and expected_window_target_h > 0.0
        and total_time_h >= SIMULATION_WINDOW_MIN_RUNTIME_FRACTION * expected_window_target_h
    )

    # 5) Continuous checks (additional hard guards for continuous modes).
    if is_continuous and np.isfinite(max_culture_age):
        all_allowed = (last_reason in CONTINUOUS_ALLOWED_TERMINATION_REASONS) and all(
            reason in CONTINUOUS_ALLOWED_TERMINATION_REASONS for reason in unique_reasons
        )
        full_runtime = total_time_h >= CONTINUOUS_MIN_RUNTIME_FRACTION * max_culture_age
        checks["continuous_allowed_termination"] = all_allowed
        checks["continuous_full_runtime"] = full_runtime
    else:
        checks["continuous_allowed_termination"] = True
        checks["continuous_full_runtime"] = True

    # 6) Acid/base fraction (a soft optimization constraint).
    total_base = float(simulated_totals.get("total_base_in_L", 0.0) or 0.0)
    total_acid = float(simulated_totals.get("total_acid_in_L", 0.0) or 0.0)
    total_medium = float(simulated_totals.get("total_medium_in_L", 0.0) or 0.0)
    acid_base_frac = (total_base + total_acid) / total_medium if total_medium > 0 else np.nan
    checks["acid_base_fraction_ok"] = (total_medium > 0) and (acid_base_frac <= MAX_ACID_BASE_FRACTION)

    # 7) Boundary CHON closure (a soft optimization constraint; unavailable
    # closure data remains non-fatal, matching ObjectiveFunction).
    closure_max_abs_pct = np.nan
    try:
        closure = data.compute_boundary_closure()
        elem_res = {element: abs(values["relative_residual_pct"]) for element, values in closure["elements"].items()}
        closure_max_abs_pct = max(elem_res.values()) if elem_res else np.nan
        checks["boundary_chon_closure_ok"] = all(value <= MAX_CLOSURE_PCT for value in elem_res.values())
    except Exception:
        checks["boundary_chon_closure_ok"] = True

    passes_filter = all(checks.values())

    return {
        "campaign": item.campaign,
        "mode_folder": item.mode_folder,
        "mode_group": item.mode_group,
        "run_id": item.run_id,
        "sweep": item.sweep,
        "export_folder": item.export_folder,
        "export_timestamp": item.export_timestamp,
        "relative_path": item.relative_path,
        "is_continuous_mode": is_continuous,
        "max_culture_age_h": max_culture_age,
        "expected_simulation_window_target_h": expected_window_target_h,
        "projection_window_h": simulated_totals.get("projection_window_h"),
        "projection_target_time_h": simulated_totals.get("projection_target_time_h"),
        "total_simulation_time_h": total_time_h,
        "last_batch_termination_reason": simulated_totals.get("last_batch_termination_reason"),
        "unique_batch_termination_reasons": " | ".join(sorted(unique_reasons)) if unique_reasons else "",
        "batch_final_volume_min_fraction": simulated_totals.get("batch_final_volume_min_fraction"),
        "batch_final_volume_max_fraction": simulated_totals.get("batch_final_volume_max_fraction"),
        "last_batch_final_volume_fraction": simulated_totals.get("last_batch_final_volume_fraction"),
        "total_base_in_L": total_base,
        "total_acid_in_L": total_acid,
        "total_medium_in_L": total_medium,
        "acid_base_fraction": acid_base_frac,
        "closure_max_abs_relative_residual_pct": closure_max_abs_pct,
        **checks,
        "passes_filter": passes_filter,
    }


evaluated_rows = [evaluate_filter_checks(item) for item in normalized_items]
results_df = pd.DataFrame(evaluated_rows)

if not results_df.empty:
    results_df = results_df.sort_values(
        ["campaign", "mode_group", "run_id", "sweep", "export_timestamp", "relative_path"]
    ).reset_index(drop=True)

print(f"Evaluated simulations: {len(results_df)}")
results_df

Evaluated simulations: 112


,campaign,mode_folder,mode_group,run_id,sweep,export_folder,export_timestamp,relative_path,is_continuous_mode,max_culture_age_h,...,closure_max_abs_relative_residual_pct,time_gte_72h,no_unwanted_termination,all_batches_final_volume_ok,all_mode_full_window_runtime,continuous_allowed_termination,continuous_full_runtime,acid_base_fraction_ok,boundary_chon_closure_ok,passes_filter
0,Campaign_1,03_Chemostat,Chemostat,chemostat,p2,chemostat_p2_20260825_0742,2026-08-25 07:42:00,Campaign_1_fixed_stoichiometry_Humbird\03_Chem...,True,960.0,...,0.000061,True,True,True,True,True,True,True,True,True
1,Campaign_1,04_ConcentratedFedBatch,Concentrated FedBatch,cf_constant,p2,cf_constant_p2_20260825_1321,2026-08-25 13:21:00,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,False,960.0,...,0.095009,True,True,True,True,True,True,True,True,True
2,Campaign_1,04_ConcentratedFedBatch,Concentrated FedBatch,cf_cspr,p2,cf_cspr_p2_20260825_2152,2026-08-25 21:52:00,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,False,960.0,...,0.089579,True,True,True,True,True,True,True,True,True
3,Campaign_1,04_ConcentratedFedBatch,Concentrated FedBatch,cf_gsp,p2,cf_gsp_p2_20260826_0417,2026-08-26 04:17:00,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,False,960.0,...,0.039451,True,True,True,True,True,True,True,True,True
4,Campaign_1,04_ConcentratedFedBatch,Concentrated FedBatch,cf_tbp,p2,cf_tbp_p2_20260826_1130,2026-08-26 11:30:00,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,False,960.0,...,0.086650,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,Campaign_6,01_FedBatch,FedBatch,fedbatch_constant,p2,fedbatch_constant_p2_20260825_0113,2026-08-25 01:13:00,Campaign_6_fixed_stoichiometry_Humbird_enhance...,False,960.0,...,0.000397,True,True,True,True,True,True,True,True,True
108,Campaign_6,01_FedBatch,FedBatch,fedbatch_csfr,p2,fedbatch_csfr_p2_20260825_0215,2026-08-25 02:15:00,Campaign_6_fixed_stoichiometry_Humbird_enhance...,False,960.0,...,0.002840,True,True,True,True,True,True,True,True,True
109,Campaign_6,01_FedBatch,FedBatch,fedbatch_exponential,p2,fedbatch_exponential_p2_20260825_0422,2026-08-25 04:22:00,Campaign_6_fixed_stoichiometry_Humbird_enhance...,False,960.0,...,0.001073,True,True,True,True,True,True,True,True,True
110,Campaign_6,01_FedBatch,FedBatch,fedbatch_gsp,p2,fedbatch_gsp_p2_20260825_1204,2026-08-25 12:04:00,Campaign_6_fixed_stoichiometry_Humbird_enhance...,False,960.0,...,0.016746,True,True,True,True,True,True,True,True,True


## 5. Reporting

In [7]:
# Pipeline counts + overall pass/fail
summary_counts = {
    "discovered_pickles": len(all_pkl_files),
    "sweep_matching_pickles": len(sweep_candidate_files),
    "selected_latest_pickles": len(selected_files),
    "loaded_objects": len(loaded),
    "load_errors": len(load_errors),
    "normalized_items": len(normalized_items),
    "normalization_errors": len(normalize_errors),
    "evaluated_rows": len(results_df),
    "passes_filter": int(results_df["passes_filter"].sum()) if not results_df.empty else 0,
    "fails_filter": int((~results_df["passes_filter"]).sum()) if not results_df.empty else 0,
}

counts_df = pd.DataFrame([summary_counts])
display(counts_df)

# Detailed table (main output)
DETAIL_COLS_FIRST = [
    "campaign",
    "mode_group",
    "run_id",
    "sweep",
    "passes_filter",
    "time_gte_72h",
    "no_unwanted_termination",
    "all_batches_final_volume_ok",
    "all_mode_full_window_runtime",
    "continuous_allowed_termination",
    "continuous_full_runtime",
    "acid_base_fraction_ok",
    "boundary_chon_closure_ok",
    "expected_simulation_window_target_h",
    "projection_window_h",
    "projection_target_time_h",
    "total_simulation_time_h",
    "batch_final_volume_min_fraction",
    "batch_final_volume_max_fraction",
    "acid_base_fraction",
    "closure_max_abs_relative_residual_pct",
    "last_batch_termination_reason",
    "unique_batch_termination_reasons",
    "export_timestamp",
    "export_folder",
    "relative_path",
]

ordered_cols = [c for c in DETAIL_COLS_FIRST if c in results_df.columns]
ordered_cols += [c for c in results_df.columns if c not in ordered_cols]

detailed_df = results_df[ordered_cols].copy() if not results_df.empty else pd.DataFrame(columns=ordered_cols)

display(detailed_df)

,discovered_pickles,sweep_matching_pickles,selected_latest_pickles,loaded_objects,load_errors,normalized_items,normalization_errors,evaluated_rows,passes_filter,fails_filter
0,197,197,112,112,0,112,0,112,112,0


,campaign,mode_group,run_id,sweep,passes_filter,time_gte_72h,no_unwanted_termination,all_batches_final_volume_ok,all_mode_full_window_runtime,continuous_allowed_termination,...,export_timestamp,export_folder,relative_path,mode_folder,is_continuous_mode,max_culture_age_h,last_batch_final_volume_fraction,total_base_in_L,total_acid_in_L,total_medium_in_L
0,Campaign_1,Chemostat,chemostat,p2,True,True,True,True,True,True,...,2026-08-25 07:42:00,chemostat_p2_20260825_0742,Campaign_1_fixed_stoichiometry_Humbird\03_Chem...,03_Chemostat,True,960.0,1.000000,317.849978,0.0,20643.799532
1,Campaign_1,Concentrated FedBatch,cf_constant,p2,True,True,True,True,True,True,...,2026-08-25 13:21:00,cf_constant_p2_20260825_1321,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,04_ConcentratedFedBatch,False,960.0,1.000000,4841.261518,0.0,314735.339021
2,Campaign_1,Concentrated FedBatch,cf_cspr,p2,True,True,True,True,True,True,...,2026-08-25 21:52:00,cf_cspr_p2_20260825_2152,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,04_ConcentratedFedBatch,False,960.0,1.000000,5869.343162,0.0,318112.110468
3,Campaign_1,Concentrated FedBatch,cf_gsp,p2,True,True,True,True,True,True,...,2026-08-26 04:17:00,cf_gsp_p2_20260826_0417,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,04_ConcentratedFedBatch,False,960.0,1.000000,6287.347318,0.0,276105.586708
4,Campaign_1,Concentrated FedBatch,cf_tbp,p2,True,True,True,True,True,True,...,2026-08-26 11:30:00,cf_tbp_p2_20260826_1130,Campaign_1_fixed_stoichiometry_Humbird\04_Conc...,04_ConcentratedFedBatch,False,960.0,1.000000,5032.316240,0.0,313554.713165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,Campaign_6,FedBatch,fedbatch_constant,p2,True,True,True,True,True,True,...,2026-08-25 01:13:00,fedbatch_constant_p2_20260825_0113,Campaign_6_fixed_stoichiometry_Humbird_enhance...,01_FedBatch,False,960.0,0.992325,143.722193,0.0,9440.922525
108,Campaign_6,FedBatch,fedbatch_csfr,p2,True,True,True,True,True,True,...,2026-08-25 02:15:00,fedbatch_csfr_p2_20260825_0215,Campaign_6_fixed_stoichiometry_Humbird_enhance...,01_FedBatch,False,960.0,0.987221,0.000000,0.0,14688.151932
109,Campaign_6,FedBatch,fedbatch_exponential,p2,True,True,True,True,True,True,...,2026-08-25 04:22:00,fedbatch_exponential_p2_20260825_0422,Campaign_6_fixed_stoichiometry_Humbird_enhance...,01_FedBatch,False,960.0,0.992971,43.209518,0.0,11442.810571
110,Campaign_6,FedBatch,fedbatch_gsp,p2,True,True,True,True,True,True,...,2026-08-25 12:04:00,fedbatch_gsp_p2_20260825_1204,Campaign_6_fixed_stoichiometry_Humbird_enhance...,01_FedBatch,False,960.0,0.986215,14.221821,0.0,9559.805638


In [8]:
# Optional summary by campaign/mode
if not detailed_df.empty:
    summary_by_mode = (
        detailed_df.groupby(["campaign", "mode_group"], as_index=False)
        .agg(
            n_runs=("passes_filter", "count"),
            n_pass=("passes_filter", "sum"),
        )
    )
    summary_by_mode["n_fail"] = summary_by_mode["n_runs"] - summary_by_mode["n_pass"]
    summary_by_mode["pass_rate_pct"] = 100.0 * summary_by_mode["n_pass"] / summary_by_mode["n_runs"]
    summary_by_mode = summary_by_mode.sort_values(["campaign", "mode_group"]).reset_index(drop=True)
else:
    summary_by_mode = pd.DataFrame(columns=["campaign", "mode_group", "n_runs", "n_pass", "n_fail", "pass_rate_pct"])

display(summary_by_mode)

,campaign,mode_group,n_runs,n_pass,n_fail,pass_rate_pct
0,Campaign_1,Chemostat,1,1,0,100.0
1,Campaign_1,Concentrated FedBatch,5,5,0,100.0
2,Campaign_1,Continuous Perfusion,5,5,0,100.0
3,Campaign_1,FedBatch,6,6,0,100.0
4,Campaign_2,Chemostat,1,1,0,100.0
5,Campaign_2,Concentrated FedBatch,5,5,0,100.0
6,Campaign_2,Continuous Perfusion,5,5,0,100.0
7,Campaign_2,FedBatch,6,6,0,100.0
8,Campaign_3,Chemostat,1,1,0,100.0
9,Campaign_3,Concentrated FedBatch,5,5,0,100.0


## 6. Export

In [9]:
if EXPORT_DETAILED_CSV and not detailed_df.empty:
    detailed_csv_path = NOTEBOOK_DIR / DETAILED_CSV_NAME
    detailed_df.to_csv(detailed_csv_path, index=False)
    print(f"Exported detailed results: {detailed_csv_path}")
else:
    print("Skipped detailed CSV export.")

if EXPORT_SUMMARY_CSV and not summary_by_mode.empty:
    summary_csv_path = NOTEBOOK_DIR / SUMMARY_CSV_NAME
    summary_by_mode.to_csv(summary_csv_path, index=False)
    print(f"Exported summary results: {summary_csv_path}")
else:
    print("Skipped summary CSV export.")

Skipped detailed CSV export.
Skipped summary CSV export.
